# 🧩 Autism Spectrum Disorder (ASD) Screening - Exploratory Data Analysis & Machine Learning

**Author:** Muttakim_A  
**Repository:** [autism_screening_ml](https://github.com/Muttakim63/autism_screening_ml)  
**Dataset:** UCI Autistic Spectrum Disorder Screening Data for Adults (704 instances, 20 features)  

---  

## 📌 Executive Summary
Autism Spectrum Disorder (ASD) is a neurodevelopmental condition characterized by differences in social communication, interaction, and behavioral patterns. Early screening allows for timely intervention and support.

In this notebook, we perform:
1. **Exploratory Data Analysis (EDA)** on AQ-10 questionnaire responses and demographic indicators.
2. **Data Preprocessing & Cleaning** (handling missing values, standardizing features, one-hot encoding).
3. **Model Benchmarking**: Evaluating 6 Machine Learning algorithms (Logistic Regression, Random Forest, Extra Trees, SVM, Gradient Boosting, AdaBoost).
4. **Hyperparameter Tuning & Serialization**: Fine-tuning the best classifier and exporting a production pipeline for web deployment.

In [ ]:
# Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# Set Plotting Aesthetics
sns.set_theme(style='darkgrid', palette='mako')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Data Ingestion & Initial Inspection

In [ ]:
# Load dataset from UCI ML Repository
autism_repo = fetch_ucirepo(id=426)
df_raw = pd.concat([autism_repo.data.features, autism_repo.data.targets], axis=1)
print(f"Raw Dataset Shape: {df_raw.shape}")
df_raw.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Load preprocessed clean dataset created by build_project.py
df_clean = pd.read_csv('../data/cleaned_autism_data.csv')
print(f"Cleaned Dataset Shape: {df_clean.shape}")
df_clean.info()

## 3. Exploratory Data Analysis (EDA)
### 3.1 Target Distribution (`class_asd`)

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df_clean, x='class_asd', palette=['#3b82f6', '#ef4444'])
plt.title('Distribution of ASD Screening Classification (0 = Negative, 1 = Positive)', fontsize=14, fontweight='bold')
plt.xlabel('Class / ASD Status')
plt.ylabel('Count of Participants')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.show()

### 3.2 AQ-10 Items Correlation Heatmap

In [ ]:
aq_cols = [f'A{i}_Score' for i in range(1, 11)] + ['result', 'class_asd']
plt.figure(figsize=(12, 8))
sns.heatmap(df_clean[aq_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('AQ-10 Score Items & Target Correlation Heatmap', fontsize=14, fontweight='bold')
plt.show()

### 3.3 Demographic Breakdown (Gender & Family History of Autism)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df_clean, x='gender', hue='class_asd', ax=axes[0], palette='magma')
axes[0].set_title('ASD Diagnosis by Gender (0 = Female, 1 = Male)', fontweight='bold')
axes[0].set_xticklabels(['Female', 'Male'])

sns.countplot(data=df_clean, x='austim', hue='class_asd', ax=axes[1], palette='viridis')
axes[1].set_title('ASD Diagnosis by Family History (austim)', fontweight='bold')
axes[1].set_xticklabels(['No Family History', 'Family History'])

plt.tight_layout()
plt.show()

## 4. Machine Learning Model Benchmarking & Selection

In [ ]:
import json
with open('../models/model_metadata.json') as f:
    metadata = json.load(f)

results_df = pd.DataFrame(metadata['benchmark']).sort_values(by='F1-Score', ascending=False)
print("=== Machine Learning Model Performance Metrics ===")
results_df

In [ ]:
# Visual Model Metric Comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='Model', y='F1-Score', palette='rocket')
plt.title('Model Benchmark Comparison (F1-Score)', fontsize=14, fontweight='bold')
plt.ylim(0.8, 1.05)
plt.xticks(rotation=15)
plt.show()

## 5. Conclusion & Deployment
- The **Random Forest** classifier achieved **100% Accuracy and 1.0 F1-Score** on test data.
- The complete feature engineering and model inference pipeline was serialized to `models/best_autism_model.joblib`.
- The model is deployed as a real-time web screening app in `app/streamlit_app.py`.